In [46]:
import numpy as np
import pandas as pd
import cvxpy as cp
import mosek
import matplotlib.pyplot as plt
import datetime as date
from datetime import datetime as dt
from dateutil.relativedelta import *
from scipy.stats import rankdata

In [64]:
def h_quad(x,m):
    return((1+m)*x-m*x**2)

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)

def robust_counterpart_chi2 (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    w = cp.Variable(N, nonneg = True)
    z = cp.Variable(M)
    s = cp.Variable(N)
    eta = cp.Variable(M, nonneg= True)
    constraints = [s/2+gamma*(np.zeros(N)+1)<= w]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append((-R @ a)[i] - lbdasum - beta - (1-cp.sum(a))*r_f <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints.append(cp.norm(cp.vstack([w[i],t[i]/2]))<=(t[i]+2*gamma)/2)
    for j in range(M):
        constraints.append(cp.norm(cp.vstack([eta[j],(z[j]-lbda[j])/2]))<=(z[j]+lbda[j])/2)
        constraints.append(1/(2*np.sqrt(m))*(-v[j]+lbda[j]+m*lbda[j])<= eta[j])
    constraints.append(cp.abs(a)<= 10)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck_chi2(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = (1+m)*cp.sum(z2)-m*cp.sum(z2)**2
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons +1/p[i]*(q[i]-p[i])**2
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)

def squeeze_algo2(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    steps = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheck_chi2(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c+1e-3:
            oldrank = np.argsort(R.dot(w))
            sets = ranktoset(oldrank)
            while nonstop:
                [w,lowerobj] = robust_counterpart_chi2(sets,p,R,r,m,r_f,c)
                newrank = np.argsort(R.dot(w))
                if np.array_equal(newrank,oldrank):
                    break
                [sets,added] = makesetflex(sets,newrank)
                oldrank = newrank
                steps = steps + 1
            return(w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        print('upperbound' , upperobj, 'cut-iterations', iterations)

In [74]:
def non_robustcheck_chi2(a,R,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    constraints = [cp.sum(q_b)==1]
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = p[rank[0:i+1]]
        v = (1+m)*cp.sum(z2)-m*cp.sum(z2)**2
        constraints.append(cp.sum(z1)-v <= 0)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)


def non_robust_cut_chi2(R,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = non_robustcheck_chi2(w,R,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c +1e-3:
            return(w,'upperbound', upperobj, 'cut-iterations', iterations)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        print('upperbound' , upperobj, 'cut-iterations', iterations)

In [48]:
df_returns6 = pd.read_csv('6_Portfolios_2x3.csv', skiprows = 15)

In [49]:
df_returns = df_returns6[0:1144].copy()
df_returns['Date'] = pd.to_datetime(df_returns['Date'], format = '%Y%m')
for i in range(1, len(df_returns.columns)):
    df_returns[df_returns.columns[i]] = pd.to_numeric(df_returns[df_returns.columns[i]])
df_returns

,Date,SMALL LoBM,ME1 BM2,SMALL HiBM,BIG LoBM,ME2 BM2,BIG HiBM
0,1926-07-01,1.0874,0.9349,-0.0695,5.7168,1.9620,1.4222
1,1926-08-01,0.7030,1.2300,5.3842,2.7154,2.6930,6.3154
2,1926-09-01,-2.9117,-0.1303,-0.4374,1.4287,0.0704,-0.7967
3,1926-10-01,-3.8196,-4.5860,-2.0112,-3.5898,-2.3398,-4.0970
4,1926-11-01,3.1806,3.7233,2.0944,3.1292,2.8952,3.4614
...,...,...,...,...,...,...,...
1139,2021-06-01,5.6058,0.4400,-1.0979,4.8188,-1.2594,-4.0036
1140,2021-07-01,-5.5593,-1.8623,-3.6521,3.1048,-0.0099,-2.3000
1141,2021-08-01,2.3903,1.5124,2.6680,3.5667,1.4122,3.0371
1142,2021-09-01,-4.3421,-3.4661,0.6445,-5.4525,-3.8570,-0.2526


In [61]:
startdate = dt(1995,1,3)
X = df_returns[df_returns.Date >= startdate][df_returns.columns[1:7]]
X = X.reset_index(drop = True)
R = X.to_numpy()

In [77]:
N=R.shape[0]
p = np.zeros(N)+1/N
I = R.shape[1]
r = 0.05
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.07
c = 10

In [78]:
squeeze_algo2(R,r,c,p,m,r_f)

rbvalue 166.5488833532528
upperbound 13.891907950318139 cut-iterations 1
rbvalue 21.2799809339678
upperbound 12.9463544131435 cut-iterations 2
rbvalue 17.493787531542925
upperbound 11.34698027522311 cut-iterations 3
rbvalue 17.806480875624356
upperbound 10.878567903447294 cut-iterations 4
rbvalue 12.34930713466363
upperbound 10.430770312230171 cut-iterations 5
rbvalue 12.606775799287949
upperbound 10.152679818509009 cut-iterations 6
rbvalue 14.055823124601
upperbound 10.137483773035552 cut-iterations 7
rbvalue 11.230739601181428
upperbound 10.05150751685796 cut-iterations 8
rbvalue 10.894058967882312
upperbound 9.962247332035709 cut-iterations 9
rbvalue 10.633305396084017
upperbound 9.870626663880406 cut-iterations 10
rbvalue 10.358149315460691
upperbound 9.76667761357702 cut-iterations 11
rbvalue 10.244712272037532
upperbound 9.736883847265318 cut-iterations 12
rbvalue 10.185545990895537
upperbound 9.72693742201488 cut-iterations 13
rbvalue 10.0936459658404
upperbound 9.70492819126570

(array([-9.23714905,  9.99999998,  3.65813019,  9.99999996, -3.41575792,
        -4.77766372]),
 'upperbound',
 9.676725496556443,
 'lowerbound',
 9.676175949468625,
 'cut-iterations',
 27,
 ' robust iterations',
 2)

In [73]:
results_nonrb = non_robust_cut_chi2(R,c,p,m,r_f)
print(results_nonrb)
robustcheck_chi2(results_nonrb[0],R,r,p,m,r_f)[0]

rbvalue 85.67052759192939
upperbound 20.72240497618754 cut-iterations 1
rbvalue 14.744315380125778
upperbound 17.649127488790377 cut-iterations 2
rbvalue 13.466294381999045
upperbound 17.409629150142457 cut-iterations 3
rbvalue 11.038432258740148
upperbound 17.15855968539835 cut-iterations 4
rbvalue 10.483893630682717
upperbound 17.012099884975086 cut-iterations 5
rbvalue 10.318094939506024
upperbound 16.89003492916256 cut-iterations 6
rbvalue 10.164542693501566
upperbound 16.8680838793589 cut-iterations 7
rbvalue 10.07789992592972
upperbound 16.804398042247232 cut-iterations 8
rbvalue 10.042023083557499
upperbound 16.803788673060602 cut-iterations 9
rbvalue 10.019832081166008
upperbound 16.800732344467765 cut-iterations 10
rbvalue 10.010729987173443
upperbound 16.800671728312363 cut-iterations 11
rbvalue 10.004845562903434
upperbound 16.79965053155821 cut-iterations 12
rbvalue 10.024616424037786
upperbound 16.799392982449366 cut-iterations 13
rbvalue 10.0065801535453
upperbound 16.799

44.33872406427757